## Object Oriented Programming: `bodaboda_business`

This notebook models a boda boda (motorcycle taxi) business as a Python class.

As a boda owner, the things that matter day-to-day are:
- **Fares** – base fare + per-km rate, with surcharges for extra weight/cargo and season (rainy season = higher demand/risk = higher fares).
- **Weight** – passenger/cargo load affects fuel use, wear on the bike, and pricing.
- **Season** – dry vs rainy season affects fuel consumption, maintenance frequency, and fare multipliers.
- **Running costs** – fuel, maintenance, insurance, loan/asset repayments.
- **Income tracking** – trips completed, total revenue, total expenses, daily/overall profit.

The `bodaboda_business` class below bundles all of this together.

In [ ]:
class bodaboda_business:
    """Models a boda boda (motorcycle taxi) business owned by a single operator."""

    SEASON_FARE_MULTIPLIER = {"dry": 1.0, "rainy": 1.3, "holiday": 1.5}
    SEASON_FUEL_MULTIPLIER = {"dry": 1.0, "rainy": 1.15, "holiday": 1.0}
    FREE_WEIGHT_KG = 20
    WEIGHT_SURCHARGE_PER_KG = 10
    MAX_LOAD_KG = 100

    def __init__(self, owner_name, bike_model, base_fare=50, rate_per_km=20,
                 fuel_price_per_litre=180, fuel_consumption_per_km=0.03,
                 season="dry"):
        self.owner_name = owner_name
        self.bike_model = bike_model
        self.base_fare = base_fare
        self.rate_per_km = rate_per_km
        self.fuel_price_per_litre = fuel_price_per_litre
        self.fuel_consumption_per_km = fuel_consumption_per_km

        if season not in self.SEASON_FARE_MULTIPLIER:
            raise ValueError(f"season must be one of {list(self.SEASON_FARE_MULTIPLIER)}")
        self.season = season

        self.weight = 0
        self.total_trips = 0
        self.total_revenue = 0.0
        self.total_expenses = 0.0
        self.expenses = []
        self.trip_log = []

    def set_season(self, season):
        """Update the current season, which affects fares and fuel usage."""
        if season not in self.SEASON_FARE_MULTIPLIER:
            raise ValueError(f"season must be one of {list(self.SEASON_FARE_MULTIPLIER)}")
        self.season = season

    def load_cargo(self, weight_kg):
        """Set the weight (passenger + goods) currently being carried."""
        if weight_kg < 0:
            raise ValueError("weight_kg cannot be negative")
        if weight_kg > self.MAX_LOAD_KG:
            raise ValueError(f"weight_kg exceeds bike's max load of {self.MAX_LOAD_KG}kg")
        self.weight = weight_kg

    def weight_surcharge(self):
        """Extra fare charged for carrying weight above the free allowance."""
        extra_kg = max(0, self.weight - self.FREE_WEIGHT_KG)
        return extra_kg * self.WEIGHT_SURCHARGE_PER_KG

    def calculate_fare(self, distance_km):
        """Total fare for a trip: base fare + distance charge + weight surcharge, seasonally adjusted."""
        distance_charge = distance_km * self.rate_per_km
        fare = self.base_fare + distance_charge + self.weight_surcharge()
        return round(fare * self.SEASON_FARE_MULTIPLIER[self.season], 2)

    def fuel_cost(self, distance_km):
        """Fuel cost for a trip, adjusted for season (e.g. rain increases consumption)."""
        litres = distance_km * self.fuel_consumption_per_km * self.SEASON_FUEL_MULTIPLIER[self.season]
        return round(litres * self.fuel_price_per_litre, 2)

    def record_trip(self, distance_km, weight_kg=None):
        """Complete a trip: charge a fare, deduct fuel cost, and log the result."""
        if weight_kg is not None:
            self.load_cargo(weight_kg)

        fare = self.calculate_fare(distance_km)
        cost = self.fuel_cost(distance_km)

        self.total_trips += 1
        self.total_revenue += fare
        self.record_expense(f"fuel (trip {self.total_trips})", cost)
        self.trip_log.append({
            "trip_number": self.total_trips,
            "distance_km": distance_km,
            "weight_kg": self.weight,
            "season": self.season,
            "fare": fare,
            "fuel_cost": cost,
        })
        return fare

    def record_expense(self, label, amount):
        """Log a business expense, e.g. maintenance, insurance, loan repayment."""
        self.expenses.append((label, amount))
        self.total_expenses += amount

    def profit(self):
        """Net profit so far: revenue earned minus all expenses."""
        return round(self.total_revenue - self.total_expenses, 2)

    def summary(self):
        """Print a snapshot of the business's performance."""
        print(f"Owner: {self.owner_name} | Bike: {self.bike_model} | Season: {self.season}")
        print(f"Trips completed: {self.total_trips}")
        print(f"Total revenue: {self.total_revenue:.2f}")
        print(f"Total expenses: {self.total_expenses:.2f}")
        print(f"Net profit: {self.profit():.2f}")

    def __repr__(self):
        return (f"bodaboda_business(owner={self.owner_name!r}, bike={self.bike_model!r}, "
                f"season={self.season!r}, trips={self.total_trips}, profit={self.profit()})")

In [11]:
# Demo: run a small boda boda business through a day of trips
biz = bodaboda_business(owner_name="Otieno", bike_model="Bajaj Boxer", season="dry")  # create the business

biz.record_trip(distance_km=5, weight_kg=10)   # normal passenger trip
biz.record_trip(distance_km=8, weight_kg=35)   # trip with extra cargo -> weight surcharge applies

biz.set_season("rainy")                        # switch season for the next trip
biz.record_trip(distance_km=6, weight_kg=15)   # rainy season -> fare & fuel multipliers applied

biz.record_expense("maintenance (oil change)", 500)  # non-trip running cost
biz.record_expense("insurance (monthly)", 1200)       # non-trip running cost

biz.summary()  # print revenue/expenses/profit snapshot
biz            # show the __repr__ of the business object

Owner: Otieno | Bike: Bajaj Boxer | Season: rainy
Trips completed: 3
Total revenue: 731.00
Total expenses: 1807.46
Net profit: -1076.46


bodaboda_business(owner='Otieno', bike='Bajaj Boxer', season='rainy', trips=3, profit=-1076.46)